# Regulatory programs and cross-modality transfer in C. elegans

This notebook follows the biological workflow from real input data to a trained model, a newly selected panel, and manuscript-matched biological analyses. It does not read packaged aggregate results as tutorial inputs. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/regulatory_section/02_SMITH_Regulatory_Activity_source.ipynb).

## Biological question

Which compact set of regulatory features is sufficient to preserve C. elegans cell identity and developmental progression? The TF and miRNA assays represent regulatory activity rather than a generic feature-selection benchmark: a useful panel should retain discrete lineage labels and the continuous developmental-time signal in held-out cells.

**How to read the endpoint:** Cell-type accuracy asks whether the panel preserves discrete lineage information, while developmental-time correlation asks whether it preserves the ordered embryonic trajectory. The paper-specific analyses then ask three biological follow-up questions: does the panel retain annotated developmental modules, can selected targets reconstruct TF co-activity in muscle, neuron, pharynx and skin, and does a panel selected from scRNA-seq retain cell identity when its genes are evaluated on held-out TF-activity lineages?

## Step 0: Download the real input data

Download the versioned Zenodo archive and verify its checksums before training:

```bash
python scripts/download_tutorial_data.py \
  --case 02_regulatory_activity \
  --data-root data/tutorials
```

The notebook is pre-executed for documentation. Read the Docs does not download large data or train SMITH during documentation builds.

The developmental-module and TF-pair tables in the archive are normalized from the source atlas Supplementary Table 5. Their preparation can be audited independently with:

```bash
python scripts/prepare_elegans_atlas_annotations.py --data-root data/tutorials
```

## Configuration

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'regulatory'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 30))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')
WORKFLOW_ENV = os.environ.copy()
WORKFLOW_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(ROOT), str(ROOT / "src"), WORKFLOW_ENV.get("PYTHONPATH", "")]
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Step 1: Inspect the biological input data

The train/test H5AD files contain lineage-aware TF or miRNA activity, cell-type labels, and absolute developmental time. The supplementary module and TF-pair annotations define the developmental programs and regulator relationships used in the manuscript analyses, while the scRNA H5AD supplies the independent reference for RNA-to-TF transfer. Training cells learn the activity representation; held-out cells test whether selected regulators still recover identity, age, and regulatory structure.

In [ ]:
inputs = ['regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/test.h5ad', 'regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv', 'regulatory_activity/elegans/annotations/tf_regulatory_pairs.tsv', 'regulatory_activity/elegans/reference/elegans_scrna.h5ad']
input_checksums = {}
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    input_checksums[relative] = sha256_file(path)


## Step 2: Train SMITH and select a panel

SMITH is trained on the training H5AD with reconstruction, cell-type classification, and developmental-time objectives. Its learned gene ranking is then truncated to the manuscript panel sizes; no packaged aggregate ranking is used.

The command below starts from the H5AD inputs above and writes a fresh model ranking, panel, evaluation, and run manifest.

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--datasets', 'elegans_tf,elegans_mirna', '--splits', 'split_1', '--methods', 'SMITH', '--seeds', '1', '--max-cells', '3000', '--paper-analyses', '--module-file', 'regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv', '--regulatory-pair-file', 'regulatory_activity/elegans/annotations/tf_regulatory_pairs.tsv', '--scrna-file', 'regulatory_activity/elegans/reference/elegans_scrna.h5ad'] + ["--force"]
completed = subprocess.run(
    command, cwd=ROOT, env=WORKFLOW_ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
if not manifest.get("training_runs"):
    raise RuntimeError("The workflow did not record any SMITH training runs.")
for relative in ['figure_data/figure3_c_f_summary.tsv', 'figure_data/figure3_c_f_paired_tests.tsv', 'figure_data/figure3_h_module_miss_rate.tsv', 'figure_data/figure3_i_coactivity.tsv', 'figure_data/figure3_j_tf_scrna_correlation.tsv', 'figure_data/figure3_k_transfer.tsv']:
    if not (CASE_OUTPUT / relative).is_file():
        raise FileNotFoundError(CASE_OUTPUT / relative)


## Step 3: Evaluate held-out biology

Cell-type accuracy asks whether the panel preserves discrete lineage information, while developmental-time correlation asks whether it preserves the ordered embryonic trajectory. The paper-specific analyses then ask three biological follow-up questions: does the panel retain annotated developmental modules, can selected targets reconstruct TF co-activity in muscle, neuron, pharynx and skin, and does a panel selected from scRNA-seq retain cell identity when its genes are evaluated on held-out TF-activity lineages?

The next cell recomputes one held-out prediction from the newly selected panel and writes truth/prediction files. The workflow also records split-level metrics and statistical metadata. No aggregate reference output is read.

In [ ]:
import warnings

from reproducibility.workflows.regulatory_activity.analysis import write_statistical_analysis
from reproducibility.workflows.regulatory_activity.evaluate_outputs import evaluate
from reproducibility.workflows.regulatory_activity.paper_analysis import (
    coactivity_reconstruction,
    tf_scrna_correlation,
    write_module_coverage,
)

panel_file = CASE_OUTPUT / "runs/elegans_tf/split_1/seed_1/panel_32/panels/SMITH_top32.tsv"
recheck_dir = CASE_OUTPUT / "notebook_recheck" / "elegans_tf"
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    evaluate(
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad",
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad",
        panel_file, recheck_dir, 32, neighbors=5,
    )
write_statistical_analysis(CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv", CASE_OUTPUT / "figure_data")
paper_outputs = manifest["outputs"]
required_paper_outputs = ("module_coverage", "coactivity", "tf_scrna_correlation", "transfer")
if any(key not in paper_outputs or not Path(paper_outputs[key]).is_file() for key in required_paper_outputs):
    raise FileNotFoundError("The paper-specific Figure 3g-k outputs were not generated")
if not (recheck_dir / "cell_type_predictions.tsv").is_file() or not (recheck_dir / "developmental_time_predictions.tsv").is_file():
    raise FileNotFoundError("Held-out prediction files were not generated")


## Step 4: Render the manuscript panels

Only the manuscript figure images are rendered below. Intermediate tables and logs remain in the output directory but are not displayed.

In [ ]:
figure_dir = CASE_OUTPUT / "figures"
plot_command = [sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/plot_figure3.py'), '--values', str(CASE_OUTPUT / 'figure_data/figure3_c_f_values.tsv'), '--modules', str(DATA_ROOT / 'regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv'), '--module-coverage', str(CASE_OUTPUT / 'figure_data/figure3_h_module_miss_rate.tsv'), '--coactivity', str(CASE_OUTPUT / 'figure_data/figure3_i_coactivity.tsv'), '--correlation', str(CASE_OUTPUT / 'figure_data/figure3_j_tf_scrna_correlation.tsv'), '--transfer', str(CASE_OUTPUT / 'figure_data/figure3_k_transfer.tsv'), "--output-dir", str(figure_dir)]
subprocess.run(
    plot_command, cwd=ROOT, env=WORKFLOW_ENV, check=True, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for heading, relative, width in [('Figure 3c - TF cell-type accuracy', 'figures/figure3_c.png', 430), ('Figure 3d - TF developmental-time correlation', 'figures/figure3_d.png', 430), ('Figure 3e - miRNA cell-type accuracy', 'figures/figure3_e.png', 430), ('Figure 3f - miRNA developmental-time correlation', 'figures/figure3_f.png', 430), ('Figure 3g - annotated developmental modules', 'figures/figure3_g.png', 500), ('Figure 3h - developmental module miss rate', 'figures/figure3_h.png', 500), ('Figure 3i - TF co-activity reconstruction', 'figures/figure3_i.png', 500), ('Figure 3j - scRNA/TF correlation structure', 'figures/figure3_j.png', 500), ('Figure 3k - scRNA-to-TF panel transfer', 'figures/figure3_k.png', 500), ('Shared method legend', 'figures/figure3_method_legend.png', 900)]:
    display(Markdown(f"### {heading}"))
    display(Image(filename=str(CASE_OUTPUT / relative), width=width))


## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--splits split_1,split_2,split_3,split_4,split_5 --methods SMITH,PERSIST-class,PERSIST,ActiveSVM,scGIST,scGeneFit,Spapros --baseline-root external/SMITH_baselines/GPS_tools-main/baselines --baseline-python PERSIST=/opt/envs/persist/bin/python --baseline-python PERSIST-class=/opt/envs/persist/bin/python --baseline-python scGIST=/opt/envs/scgist/bin/python --epochs 200
```

This executed page uses one real TF split, one real miRNA split and one scRNA-to-TF transfer split. The paper command above regenerates Figure 3c-k with all five lineage-aware splits and manuscript baselines. The module, TF-pair and scRNA inputs are versioned biological inputs; the workflow stops with an explicit error if they are absent. This quick hosted run is intentionally smaller than the paper-scale comparison, but it starts from the same real inputs and executes the same prediction and analysis functions.